# The learning loop: twelve points of training data

In the lecture, a line found its way through twelve points: a guess, the loss, a step downhill,
again and again. In this notebook you run that loop yourself and turn its knobs: the step size,
the number of parameters, the amount of training data. A model with eight parameters then
learns its training data by heart, validation data show it, and the three sets from the lecture
get their jobs.

**Keep your work.** In Colab, choose *File → Save a copy in Drive* first: changes to the copy
opened from GitHub are not saved.

**How to work.** Run one cell at a time with Shift+Enter, make adjustments, experiment.

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt

%config InlineBackend.figure_format = "retina"   # sharp pictures on high-resolution screens

SEED = 2026
random.seed(SEED)

def draw(x, y, curve_x=None, curve_y=None, val_x=None, val_y=None, title=""):
    """The training data as dots; a model as a curve, if given; validation data as open circles, if given."""
    plt.figure(figsize=(6, 4))
    plt.scatter(x, y, color="black", zorder=3, label="training data")
    if curve_x is not None:
        plt.plot(curve_x, curve_y, color="tab:blue", label="the model")
    if val_x is not None:
        plt.scatter(val_x, val_y, facecolors="none", edgecolors="tab:red", zorder=3, label="validation data")
    plt.xlim(-1.1, 1.1)
    plt.ylim(-2, 3)
    plt.xlabel("input x")
    plt.ylabel("output y")
    plt.title(title)
    plt.legend(loc="upper left")
    plt.show()

def loss(outputs, desired):
    """The gaps between the outputs and the desired outputs, squared and added up."""
    return ((outputs - desired) ** 2).sum()

## 1. Twelve points of training data

The same twelve points as on yesterday's slide: each is an input `x` with its desired output
`y`, the label.

In [ ]:
x = np.array([-0.991, -0.804, -0.574, -0.424, -0.300, -0.093, 0.084, 0.225, 0.455, 0.566, 0.766, 0.952])
y = np.array([-1.167, -1.012, -0.800, -0.402, 0.015, 0.103, 0.686, 0.559, 0.944, 1.525, 1.526, 1.491])
print(len(x), "points of training data; the first: input", x[0], "desired output", y[0])
draw(x, y, title="twelve points of training data")

## 2. A line, two parameters

The model is a straight line, `y = a * x + b`: two adjustable parameters, the slope `a` and the
offset `b`. The first guess slopes the wrong way. Its loss is the gap between the line
and each desired output, squared and added up.

In [ ]:
a, b = -0.6, 1.3                    # the first guess from the slide
outputs = a * x + b
print("loss", round(loss(outputs, y), 2))
grid = np.linspace(-1, 1, 100)      # inputs for drawing a model
draw(x, y, grid, a * grid + b, title=f"a = {a}, b = {b}: loss {loss(outputs, y):.2f}")

In [ ]:
# Your turn: set a and b by hand until the loss is below 1. Change the two numbers, run, look.
a, b = -0.6, 1.3
outputs = a * x + b
draw(x, y, grid, a * grid + b, title=f"a = {a}, b = {b}: loss {loss(outputs, y):.2f}")

## 3. The step downhill

Nobody reasons out `a` and `b` in practice, the supervised learning loop does. At the current `a`
and `b` the computer works out how and 'in which direction' (i.e., making them smaller or larger)
to change `a` and `b`.
The magnitude of the change is called the learning rate. The two lines that give the slope come
from calculus; you need not derive them, only watch what they do.

In [ ]:
LEARNING_RATE = 0.0375   # the step size; the slide's value
STEPS = 20

a, b = -0.6, 1.3
losses = np.zeros(STEPS + 1)
losses[0] = loss(a * x + b, y)
for step in range(1, STEPS + 1):
    gaps = a * x + b - y                     # compare: the line's outputs against the desired ones
    slope_a = 2 * (gaps * x).sum()           # which way to improve, for a
    slope_b = 2 * gaps.sum()                 # and for b
    a = a - LEARNING_RATE * slope_a          # adjust: a small step downhill
    b = b - LEARNING_RATE * slope_b
    losses[step] = loss(a * x + b, y)
    print(f"step {step:2d}: a {a:+.3f}  b {b:+.3f}  loss {losses[step]:.3f}")
draw(x, y, grid, a * grid + b, title=f"after {STEPS} steps: a = {a:.2f}, b = {b:.2f}, loss {losses[STEPS]:.2f}")

In [ ]:
# The loss after each step: big improvement at first, then slowly, then no further. The slide's
# 9.5, 4.5 and 0.8 are steps 1, 2 and 5.
plt.figure(figsize=(6, 3.5))
plt.plot(range(STEPS + 1), losses, marker="o")
plt.xlabel("step")
plt.ylabel("loss")
plt.show()

In [ ]:
# Your turn: the learning rate. Find a value at which the steps overshoot and the loss grows
# instead of falling, and one at which it crawls. Change LEARNING_RATE, run, read the losses.
LEARNING_RATE = 0.0375
a, b = -0.6, 1.3
losses = np.zeros(STEPS + 1)
losses[0] = loss(a * x + b, y)
for step in range(1, STEPS + 1):
    gaps = a * x + b - y
    a = a - LEARNING_RATE * 2 * (gaps * x).sum()
    b = b - LEARNING_RATE * 2 * gaps.sum()
    losses[step] = loss(a * x + b, y)
    print(f"step {step:2d}: loss {losses[step]:.3f}")
plt.figure(figsize=(6, 3.5))
plt.plot(range(STEPS + 1), losses, marker="o")
plt.xlabel("step")
plt.ylabel("loss")
plt.title(f"learning rate {LEARNING_RATE}")
plt.show()

## 4. Eight parameters

Yesterday's second picture: eight points of training data from the same source, and a more
complex model than a line. The same loop trains it, from all parameters at zero. A million steps
take a few seconds; the cell keeps the parameters at a few checkpoints on the way.

In [ ]:
x8 = np.array([-0.927, -0.676, -0.455, -0.162, 0.150, 0.399, 0.711, 0.985])
y8 = np.array([-0.824, -0.959, -0.365, 0.468, 0.443, 0.181, 1.607, 1.667])

def powers(x):
    """The inputs raised to the powers 0 to 7, one column each: what the eight parameters multiply."""
    return np.stack([x ** k for k in range(8)], axis=1)

def outputs_of(parameters, x):
    """The model's outputs: each power of x times its parameter, added up."""
    return powers(x) @ parameters

CHECKPOINTS = [1, 10, 100, 1000, 10000, 100000, 1000000]
LEARNING_RATE = 0.044

inputs = powers(x8)
parameters = np.zeros(8)
kept = np.zeros((len(CHECKPOINTS), 8))         # the parameters at each checkpoint
train_loss = np.zeros(len(CHECKPOINTS))
for step in range(1, CHECKPOINTS[-1] + 1):
    gaps = inputs @ parameters - y8            # compare
    slope = 2 * (inputs.T @ gaps)              # which way is down, for all eight parameters at once
    parameters = parameters - LEARNING_RATE * slope   # adjust
    if step in CHECKPOINTS:
        c = CHECKPOINTS.index(step)
        kept[c] = parameters
        train_loss[c] = loss(inputs @ parameters, y8)
        print(f"step {step:8d}: training loss {train_loss[c]:.3f}")
draw(x8, y8, grid, outputs_of(parameters, grid), title=f"after a million steps: training loss {train_loss[-1]:.3f}")

A perfect approximation of the data: the curve passes through every point, training loss zero.
Perfect?

## 5. Validation data

Six more points from the same source, which the model never learns from: the validation data.
The loss on them is the validation loss. The table and the chart give both losses at every
checkpoint.

In [ ]:
x_val = np.array([-0.841, -0.296, -0.038, 0.323, 0.560, 0.799])
y_val = np.array([-0.613, -0.025, -0.039, 0.822, 0.945, 1.587])
validation_loss = np.zeros(len(CHECKPOINTS))
for c in range(len(CHECKPOINTS)):
    validation_loss[c] = loss(outputs_of(kept[c], x_val), y_val)
    print(f"step {CHECKPOINTS[c]:8d}: training loss {train_loss[c]:.3f}   validation loss {validation_loss[c]:.3f}")
plt.figure(figsize=(6, 3.5))
plt.plot(CHECKPOINTS, train_loss, marker="o", label="training loss")
plt.plot(CHECKPOINTS, validation_loss, marker="o", label="validation loss")
plt.xscale("log")
plt.xlabel("step")
plt.ylabel("loss")
plt.legend()
plt.show()

In [ ]:
# Your turn: which version to keep? Set KEEP to the checkpoint whose parameters you would use,
# run, and see that version with the training and the validation data.
KEEP = 10000
c = CHECKPOINTS.index(KEEP)
draw(x8, y8, grid, outputs_of(kept[c], grid), x_val, y_val,
     title=f"step {KEEP}: training loss {train_loss[c]:.2f}, validation loss {validation_loss[c]:.2f}")

## 6. More training data

The lecture's rule of thumb: the more complex the model, the more training data is needed. You
can experiment below with the amount of training data for a more complex model. Twenty points
from the same source are the validation data. Each loss is divided by its number of points, so
that sets of different sizes compare.

In [ ]:
# The source: a curve of the same kind as the model, its own eight parameters p0 to p7.
source = np.array([0.5, 3.0, 0.0, -3.1, 0.0, 1.0, 0.0, 0.0])

def generate_data(n, seed):
    """n points from the source: inputs at random between -1 and 1, outputs on the source's curve plus noise. The same seed gives the same points; a larger n keeps them and adds more."""
    random.seed(seed)
    x = np.zeros(n)
    noise = np.zeros(n)
    for i in range(n):
        x[i] = random.uniform(-1, 1)
        noise[i] = random.gauss(0, 0.28)
    return x, outputs_of(source, x) + noise

x_val_more, y_val_more = generate_data(20, seed=3)   # the validation data for this section
print(len(x_val_more), "points of validation data; the first: input", x_val_more[0].round(3), "desired output", y_val_more[0].round(3))

In [ ]:
# Your turn: raise N_MORE until the validation loss no longer rises towards the end.
N_MORE = 8
x_more, y_more = generate_data(N_MORE, seed=6)   # the training data: other points than the validation data
inputs = powers(x_more)
learning_rate = 0.35 / N_MORE   # the loss adds up over the points: more of them, a steeper valley, smaller steps
parameters = np.zeros(8)
for step in range(1, 100001):
    gaps = inputs @ parameters - y_more
    parameters = parameters - learning_rate * 2 * (inputs.T @ gaps)
    if step in CHECKPOINTS:
        print(f"step {step:7d}: training loss per point {loss(inputs @ parameters, y_more) / N_MORE:.3f}   "
              f"validation loss per point {loss(outputs_of(parameters, x_val_more), y_val_more) / len(x_val_more):.3f}")
draw(x_more, y_more, grid, outputs_of(parameters, grid), x_val_more, y_val_more,
     title=f"{N_MORE} points of training data, eight parameters")